# Titanic Survival Prediction — Hyperparameter Tuning

## Objective

Use GridSearchCV to find suitable Logistic Regression
hyperparameters through cross-validation.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split

train_df = pd.read_csv("../data/raw/train.csv")

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X = train_df[features]
y = train_df["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

Training shape: (712, 7)
Validation shape: (179, 7)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numerical_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = ["Sex", "Embarked"]

numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Pipeline created successfully!")

Pipeline created successfully!


In [3]:
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

print(param_grid)

{'classifier__C': [0.01, 0.1, 1, 10, 100]}


In [4]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Grid Search completed!")
print("Best parameters:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.2%}")

Grid Search completed!
Best parameters: {'classifier__C': 0.1}
Best CV accuracy: 80.20%


## Evaluating the Tuned Model

Evaluate the best model selected by GridSearchCV
on the held-out validation dataset.

In [5]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_val)

tuned_accuracy = accuracy_score(y_val, y_pred_tuned)

print(f"Tuned Model Accuracy: {tuned_accuracy:.2%}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred_tuned))

Tuned Model Accuracy: 79.33%

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       110
           1       0.78      0.65      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179


Confusion Matrix:
[[97 13]
 [24 45]]


In [6]:
original_accuracy = 0.8045

print(f"Original Model: {original_accuracy:.2%}")
print(f"Tuned Model:    {tuned_accuracy:.2%}")

difference = (tuned_accuracy - original_accuracy) * 100

print(f"Difference: {difference:+.2f} percentage points")

Original Model: 80.45%
Tuned Model:    79.33%
Difference: -1.12 percentage points


In [7]:
cv_results = pd.DataFrame(grid_search.cv_results_)

comparison = cv_results[
    ["param_classifier__C", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")

print(comparison)

   param_classifier__C  mean_test_score  std_test_score  rank_test_score
1                 0.10         0.802039        0.023003                1
2                 1.00         0.796405        0.019225                2
3                10.00         0.792190        0.021465                3
4               100.00         0.792190        0.021465                3
0                 0.01         0.766936        0.032396                5


## Hyperparameter Tuning — Results and Conclusions

We used GridSearchCV with five-fold cross-validation to tune
the regularization parameter C of Logistic Regression.

### Results

- Best hyperparameter: C = 0.1
- Best cross-validation accuracy: 80.20%
- Tuned model validation accuracy: 79.33%
- Original model validation accuracy: 80.45%
- Majority-class baseline accuracy: 61.45%

### Conclusions

GridSearchCV selected C = 0.1 based on cross-validation
performance. However, the tuned model performed slightly worse
than the original model on the held-out validation set.

The original model correctly classified 144 of 179 passengers,
while the tuned model correctly classified 142.

This experiment demonstrates that hyperparameter tuning does
not guarantee improved validation performance.

The difference between the two models is small, so additional
cross-validation or independent evaluation would be needed
to establish which configuration generalizes better.